<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/Instrumento_de_riesgo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ================================================================
# LABORATORIO INTERACTIVO DE RIESGOS OPERACIONALES
# MBA - EXCELENCIA OPERACIONAL
# ================================================================
#
# OBJETIVO:
# Identificar, evaluar, priorizar y tratar riesgos asociados
# a un problema operacional definido por cada equipo.
#
# METODOLOGÍA:
# 1. Contextualización de la empresa y del problema.
# 2. Identificación de riesgos y causas.
# 3. Evaluación de probabilidad e impacto.
# 4. Evaluación de controles y riesgo residual esperado.
# 5. Construcción de matrices de calor.
# 6. Priorización y análisis de categorías.
# 7. Interpretaciones gerenciales.
# 8. Exportación de resultados a Excel.
#
# NOTA:
# La escala de 1 a 5 es ordinal y semicuantitativa.
# Las puntuaciones no representan probabilidades reales
# ni estimaciones monetarias de pérdidas.
# ================================================================

import sys
import subprocess
import importlib.util

# Instalación automática de dependencias faltantes.
for paquete in ["pandas", "numpy", "matplotlib", "openpyxl"]:
    if importlib.util.find_spec(paquete) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", paquete]
        )

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. CONFIGURACIÓN GENERAL
# ================================================================

# Categorías aplicables a empresas de servicios y manufactura.
CATEGORIAS = [
    "Procesos",
    "Personas",
    "Tecnología",
    "Proveedores / cadena de suministro",
    "Financiero",
    "Legal / cumplimiento",
    "Seguridad y salud",
    "Ambiental / sostenibilidad",
    "Reputacional / clientes",
    "Otro"
]

# Colores de clasificación de riesgos.
COLORES = {
    "Bajo": "#35A853",
    "Moderado": "#F5D547",
    "Alto": "#F59E42",
    "Crítico": "#D83A48"
}


def nivel(puntaje):
    """
    Clasificación del riesgo según su puntuación.

    Los umbrales corresponden a una convención didáctica
    que puede adaptarse a la política de cada organización.
    """
    if puntaje <= 4:
        return "Bajo"
    elif puntaje <= 9:
        return "Moderado"
    elif puntaje <= 16:
        return "Alto"
    else:
        return "Crítico"


# ================================================================
# 2. FUNCIONES PARA REALIZAR PREGUNTAS
# ================================================================

def pedir_texto(pregunta, obligatorio=True, defecto=""):
    """
    Solicita información textual.

    Se verifica que las preguntas obligatorias tengan
    respuesta antes de continuar con el cuestionario.
    """
    while True:
        mensaje = pregunta + " "
        if defecto:
            mensaje += f"[{defecto}] "

        respuesta = input(mensaje).strip()

        if respuesta:
            return respuesta

        if defecto:
            return defecto

        if not obligatorio:
            return ""

        print("Se requiere una respuesta para continuar.")


def pedir_entero(pregunta, minimo, maximo, defecto=None):
    """
    Solicita un número entero dentro de un rango válido.

    Esta validación evita puntuaciones fuera de la
    escala de probabilidad e impacto.
    """
    while True:
        mensaje = f"{pregunta} ({minimo}-{maximo})"

        if defecto is not None:
            mensaje += f" [{defecto}]"

        texto = input(mensaje + ": ").strip()

        if not texto and defecto is not None:
            return defecto

        try:
            numero = int(texto)
            if minimo <= numero <= maximo:
                return numero
        except ValueError:
            pass

        print(
            f"Introducir un número entero "
            f"entre {minimo} y {maximo}."
        )


def elegir_categoria():
    """
    Presenta las categorías disponibles y permite
    seleccionar la correspondiente al riesgo.
    """
    print("\nCategorías de riesgo:")

    for i, categoria in enumerate(CATEGORIAS, 1):
        print(f"  {i}. {categoria}")

    seleccion = pedir_entero(
        "Número de categoría",
        1,
        len(CATEGORIAS)
    )

    return CATEGORIAS[seleccion - 1]


def explicar_escala():
    """
    Presenta las escalas de valoración.

    La probabilidad y el impacto deben justificarse
    mediante evidencia, experiencia o supuestos.
    """
    print("\nESCALA DE PROBABILIDAD")
    print("1 = Rara")
    print("2 = Improbable")
    print("3 = Posible")
    print("4 = Probable")
    print("5 = Casi segura")

    print("\nESCALA DE IMPACTO")
    print("1 = Insignificante")
    print("2 = Menor")
    print("3 = Moderado")
    print("4 = Mayor")
    print("5 = Severo")

    print(
        "\nIMPORTANTE: estas valoraciones son ordinales. "
        "No equivalen directamente a porcentajes "
        "de probabilidad.\n"
    )


# ================================================================
# 3. CUESTIONARIO INTERACTIVO
# ================================================================

def registrar():

    print("\n" + "=" * 65)
    print("LABORATORIO DE RIESGOS OPERACIONALES")
    print("=" * 65)

    # ------------------------------------------------------------
    # ETAPA 1. CONTEXTO DEL PROYECTO
    # ------------------------------------------------------------

    print("\nETAPA 1. CONTEXTO DE LA EMPRESA")

    empresa = pedir_texto(
        "¿Cuál es la empresa o el proyecto?"
    )

    sector = pedir_texto(
        "¿A qué sector económico pertenece?"
    )

    problema = pedir_texto(
        "¿Cuál es el problema operacional central?"
    )

    objetivo = pedir_texto(
        "¿Qué objetivo estratégico se relaciona con el problema?"
    )

    proceso = pedir_texto(
        "¿Qué proceso se encuentra bajo estudio?"
    )

    cantidad = pedir_entero(
        "¿Cuántos riesgos se analizarán?",
        3,
        20,
        5
    )

    explicar_escala()

    registros = []

    # ------------------------------------------------------------
    # ETAPA 2. IDENTIFICACIÓN Y EVALUACIÓN
    # ------------------------------------------------------------

    for i in range(1, cantidad + 1):

        print("\n" + "-" * 65)
        print(f"RIESGO {i} DE {cantidad}")
        print("-" * 65)

        nombre = pedir_texto(
            "¿Cuál es el nombre breve del riesgo?"
        )

        categoria = elegir_categoria()

        causa = pedir_texto(
            "¿Cuáles son las principales causas del riesgo?"
        )

        consecuencia = pedir_texto(
            "¿Qué consecuencias tendría para la operación?"
        )

        evidencia = pedir_texto(
            "¿Qué evidencia o supuesto sustenta la valoración?"
        )

        # Riesgo inherente: exposición antes de los controles
        # considerados en el escenario de tratamiento.
        p = pedir_entero(
            "Probabilidad inherente",
            1,
            5
        )

        imp = pedir_entero(
            "Impacto inherente",
            1,
            5
        )

        puntaje = p * imp

        print("\nRESULTADO INICIAL")
        print(f"Puntuación inherente: {puntaje}/25")
        print(f"Clasificación: {nivel(puntaje)}")

        # --------------------------------------------------------
        # ETAPA 3. CONTROLES Y RIESGO RESIDUAL
        # --------------------------------------------------------

        controles = pedir_texto(
            "¿Qué controles actuales o propuestos existen?"
        )

        print(
            "\nSe evaluará el escenario residual esperado "
            "después de los controles considerados."
        )

        pr = pedir_entero(
            "Probabilidad residual esperada",
            1,
            5,
            p
        )

        ir = pedir_entero(
            "Impacto residual esperado",
            1,
            5,
            imp
        )

        if pr > p or ir > imp:
            print(
                "AVISO: el escenario residual empeora "
                "en alguna dimensión. Revisar los supuestos."
            )

        residual = pr * ir

        responsable = pedir_texto(
            "¿Qué cargo será responsable del seguimiento?"
        )

        indicador = pedir_texto(
            "¿Qué indicador de alerta temprana (KRI) se utilizará?"
        )

        umbral = pedir_texto(
            "¿Cuál será el umbral de alerta del indicador?"
        )

        accion = pedir_texto(
            "¿Qué acción adicional se recomienda implementar?"
        )

        plazo = pedir_texto(
            "¿Cuál es el plazo propuesto?"
        )

        costo = pedir_texto(
            "¿Qué recursos o costos se estiman?",
            defecto="Pendiente"
        )

        # La reducción porcentual corresponde únicamente
        # a la variación de las puntuaciones ordinales.
        # No representa una reducción real de pérdidas.
        reduccion = 100 * (puntaje - residual) / puntaje

        registros.append({
            "ID": f"R{i:02d}",
            "Riesgo": nombre,
            "Categoría": categoria,
            "Causas": causa,
            "Consecuencias": consecuencia,
            "Evidencia": evidencia,
            "Probabilidad": p,
            "Impacto": imp,
            "Puntaje_inherente": puntaje,
            "Nivel_inherente": nivel(puntaje),
            "Controles": controles,
            "Probabilidad_residual": pr,
            "Impacto_residual": ir,
            "Puntaje_residual": residual,
            "Nivel_residual": nivel(residual),
            "Reducción_puntos": puntaje - residual,
            "Reducción_porcentual": round(reduccion, 1),
            "Responsable": responsable,
            "KRI": indicador,
            "Umbral": umbral,
            "Acción": accion,
            "Plazo": plazo,
            "Recursos": costo
        })

    contexto = {
        "Empresa": empresa,
        "Sector": sector,
        "Problema": problema,
        "Objetivo": objetivo,
        "Proceso": proceso
    }

    return contexto, pd.DataFrame(registros)


# ================================================================
# 4. INTERPRETACIÓN GERENCIAL AUTOMÁTICA
# ================================================================

def interpretar(df, meta):
    """
    Genera interpretaciones basadas en los datos registrados.

    Se diferencia entre:
    - Exposición inherente.
    - Exposición residual esperada.
    - Priorización de riesgos.
    - Concentración por categorías.
    - Recomendaciones gerenciales.

    Las conclusiones no deben interpretarse como
    resultados causalmente demostrados.
    """

    n = len(df)

    criticos = df[
        df["Nivel_inherente"] == "Crítico"
    ]

    prioritarios = df[
        df["Nivel_inherente"].isin(["Crítico", "Alto"])
    ]

    principal = df.sort_values(
        ["Puntaje_inherente", "Impacto"],
        ascending=False
    ).iloc[0]

    suma_inherente = df["Puntaje_inherente"].sum()
    suma_residual = df["Puntaje_residual"].sum()

    variacion = (
        (suma_inherente - suma_residual)
        / suma_inherente * 100
    )

    persistentes = df[
        df["Nivel_residual"].isin(["Crítico", "Alto"])
    ]

    categorias = (
        df.groupby("Categoría")["Puntaje_inherente"]
        .sum()
        .sort_values(ascending=False)
    )

    texto = f"""
===============================================================
INFORME DE INTERPRETACIÓN GERENCIAL
===============================================================

EMPRESA: {meta['Empresa']}
PROBLEMA: {meta['Problema']}
OBJETIVO ESTRATÉGICO: {meta['Objetivo']}

---------------------------------------------------------------
1. INTERPRETACIÓN DE LA MATRIZ DE CALOR INHERENTE
---------------------------------------------------------------

Se identificaron {n} riesgos operacionales.

De ellos, {len(criticos)} fueron clasificados como críticos y
{len(prioritarios)} se encuentran en niveles altos o críticos.

La matriz permite observar qué riesgos combinan valores
elevados de probabilidad e impacto.

Los riesgos situados en las zonas rojas requieren especial
atención gerencial porque su puntuación indica una exposición
elevada según la metodología utilizada.

Sin embargo, la clasificación no significa que el evento
ocurrirá necesariamente ni permite calcular su frecuencia real.

Se recomienda revisar las causas, los datos disponibles y las
consecuencias de los riesgos más relevantes.

---------------------------------------------------------------
2. INTERPRETACIÓN DEL RANKING DE RIESGOS
---------------------------------------------------------------

El riesgo con mayor puntuación inherente es:

Código: {principal['ID']}
Riesgo: {principal['Riesgo']}
Probabilidad: {principal['Probabilidad']}/5
Impacto: {principal['Impacto']}/5
Puntuación: {principal['Puntaje_inherente']}/25
Nivel: {principal['Nivel_inherente']}

Este riesgo constituye una prioridad preliminar de análisis.

Se recomienda verificar si sus causas están relacionadas
con variabilidad, sobrecarga, desperdicios, fallas de proceso,
limitaciones tecnológicas o problemas de coordinación.

La priorización definitiva debe considerar también la
criticidad regulatoria, la seguridad, los costos y la
factibilidad de las alternativas de tratamiento.

---------------------------------------------------------------
3. INTERPRETACIÓN DEL RIESGO RESIDUAL
---------------------------------------------------------------

Suma de puntuaciones inherentes: {suma_inherente}
Suma de puntuaciones residuales: {suma_residual}

Variación relativa de puntuaciones: {variacion:.1f}%

La comparación muestra el cambio esperado en las
puntuaciones después de los controles considerados.

Esta variación no representa una disminución demostrada
de pérdidas económicas, frecuencia de fallas o
probabilidad real de ocurrencia.

Los resultados residuales son escenarios estimados que
deben validarse después de implementar los controles.

Permanecen {len(persistentes)} riesgos en nivel alto o crítico.

Estos riesgos requieren revisión gerencial, tratamiento
adicional o aceptación formal de acuerdo con la política
de riesgos de la organización.

---------------------------------------------------------------
4. INTERPRETACIÓN POR CATEGORÍAS
---------------------------------------------------------------

La categoría con mayor suma de puntuaciones es:

{categorias.index[0]}

Este resultado sugiere revisar la concentración de riesgos
registrados en dicha categoría.

Debe considerarse que una categoría puede presentar una
suma elevada simplemente porque contiene más riesgos.

Por tanto, esta gráfica sirve para explorar la distribución
del registro, pero no constituye una medición estandarizada
de la exposición entre departamentos.

---------------------------------------------------------------
5. RECOMENDACIONES GERENCIALES
---------------------------------------------------------------

Se recomienda:

- Revisar prioritariamente los riesgos altos y críticos.
- Verificar la evidencia utilizada para calificarlos.
- Establecer responsables y fechas de tratamiento.
- Definir indicadores de alerta temprana.
- Evaluar los costos y beneficios de los controles.
- Validar periódicamente el riesgo residual.
- Relacionar las decisiones con el objetivo estratégico.

Los resultados pueden utilizarse como insumo para
herramientas AHP, DMAIC, Vensim y análisis de escenarios.

---------------------------------------------------------------
6. LIMITACIONES METODOLÓGICAS
---------------------------------------------------------------

La metodología utiliza escalas ordinales de 1 a 5.

Los umbrales son convenciones didácticas.

El producto probabilidad × impacto es un mecanismo de
priorización y no una estimación de pérdida esperada.

La evaluación no reemplaza los estudios especializados
de seguridad, cumplimiento o exposición financiera.
"""

    if (df["Reducción_puntos"] < 0).any():
        texto += """
ADVERTENCIA:
Se encontraron riesgos cuya puntuación residual supera
la inherente. Deben revisarse los supuestos utilizados.
"""

    return texto


# ================================================================
# 5. MATRIZ DE CALOR 5 × 5
# ================================================================

def matriz(ax, df, prefijo, titulo):
    """
    Construye una matriz de calor de cinco niveles de
    probabilidad y cinco niveles de impacto.

    Cada celda presenta:
    - Puntuación de riesgo.
    - Identificadores de los riesgos ubicados en ella.

    La posición de los riesgos se obtiene de las respuestas
    proporcionadas por los estudiantes.
    """

    valores = np.array([
        [p * i for p in range(1, 6)]
        for i in range(1, 6)
    ])

    mapa = np.array([
        [
            0 if valor <= 4
            else 1 if valor <= 9
            else 2 if valor <= 16
            else 3
            for valor in fila
        ]
        for fila in valores
    ])

    cmap = ListedColormap([
        COLORES["Bajo"],
        COLORES["Moderado"],
        COLORES["Alto"],
        COLORES["Crítico"]
    ])

    norm = BoundaryNorm(
        np.arange(-0.5, 4.5, 1),
        4
    )

    ax.imshow(
        mapa,
        cmap=cmap,
        norm=norm,
        origin="lower",
        extent=[0.5, 5.5, 0.5, 5.5],
        aspect="equal"
    )

    ax.set_xticks(range(1, 6))
    ax.set_yticks(range(1, 6))

    ax.set_xlabel("Impacto (1–5)")
    ax.set_ylabel("Probabilidad (1–5)")
    ax.set_title(titulo, fontweight="bold")

    # Se identifican los riesgos correspondientes a cada celda.
    for p in range(1, 6):
        for i in range(1, 6):

            ids = df.loc[
                (df[prefijo + "Probabilidad"] == p)
                &
                (df[prefijo + "Impacto"] == i),
                "ID"
            ].tolist()

            texto = f"{p * i}\n"

            if ids:
                texto += ", ".join(ids)

            ax.text(
                i,
                p,
                texto,
                ha="center",
                va="center",
                fontsize=8 if ids else 10,
                color="#202124",
                fontweight="bold" if ids else "normal"
            )

    ax.set_xticks(
        np.arange(0.5, 6, 1),
        minor=True
    )

    ax.set_yticks(
        np.arange(0.5, 6, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=2
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )


# ================================================================
# 6. GENERACIÓN DE GRÁFICAS
# ================================================================

def graficas(df, ruta):
    """
    Produce cuatro archivos gráficos:

    01. Matrices de calor inherente y residual.
    02. Ranking de riesgos.
    03. Comparación inherente frente a residual.
    04. Distribución por categorías y niveles.

    Las gráficas utilizan los datos introducidos durante
    el cuestionario y se guardan en formato PNG.
    """

    plt.rcParams.update({
        "figure.dpi": 130,
        "savefig.dpi": 160,
        "font.size": 10
    })

    # ------------------------------------------------------------
    # GRÁFICA 1. MATRICES DE CALOR
    # ------------------------------------------------------------

    fig, axs = plt.subplots(
        1,
        2,
        figsize=(13, 6),
        constrained_layout=True
    )

    matriz(
        axs[0],
        df,
        "",
        "Riesgo inherente (antes de controles)"
    )

    matriz(
        axs[1],
        df,
        "_residual",
        "Riesgo residual esperado (después)"
    )

    fig.legend(
        handles=[
            Patch(color=color, label=nombre)
            for nombre, color in COLORES.items()
        ],
        loc="lower center",
        ncol=4,
        bbox_to_anchor=(0.5, -0.05)
    )

    fig.savefig(
        ruta / "01_matrices_calor.png",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # ------------------------------------------------------------
    # GRÁFICA 2. RANKING DE RIESGOS
    # ------------------------------------------------------------

    orden = df.sort_values("Puntaje_inherente")

    fig, ax = plt.subplots(
        figsize=(12, max(5, len(df) * 0.46 + 2)),
        layout="constrained"
    )

    colores = [
        COLORES[n]
        for n in orden["Nivel_inherente"]
    ]

    barras = ax.barh(
        orden["ID"] + " · " + orden["Riesgo"].str.slice(0, 36),
        orden["Puntaje_inherente"],
        color=colores
    )

    ax.bar_label(barras, padding=3)

    ax.set_xlim(0, 27)
    ax.set_xlabel(
        "Puntuación ordinal (probabilidad × impacto)"
    )
    ax.set_title(
        "Ranking de riesgos inherentes",
        fontweight="bold"
    )

    ax.grid(axis="x", alpha=0.2)

    fig.savefig(
        ruta / "02_ranking_riesgos.png"
    )

    plt.show()
    plt.close(fig)

    # ------------------------------------------------------------
    # GRÁFICA 3. INHERENTE FRENTE A RESIDUAL
    # ------------------------------------------------------------

    y = np.arange(len(orden))

    fig, ax = plt.subplots(
        figsize=(12, max(5, len(df) * 0.48 + 2)),
        layout="constrained"
    )

    ax.barh(
        y - 0.19,
        orden["Puntaje_inherente"],
        height=0.36,
        label="Inherente",
        color="#DB5966"
    )

    ax.barh(
        y + 0.19,
        orden["Puntaje_residual"],
        height=0.36,
        label="Residual esperado",
        color="#268E9A"
    )

    etiquetas = (
        orden["ID"] + " · "
        + orden["Riesgo"].str.slice(0, 35)
    )

    ax.set_yticks(y, etiquetas)
    ax.set_xlim(0, 27)
    ax.set_xlabel("Puntuación ordinal /25")
    ax.set_title(
        "Comparación antes y después de controles",
        fontweight="bold"
    )

    ax.legend()
    ax.grid(axis="x", alpha=0.2)

    fig.savefig(
        ruta / "03_inherente_vs_residual.png"
    )

    plt.show()
    plt.close(fig)

    # ------------------------------------------------------------
    # GRÁFICA 4. PERFIL DE RIESGOS
    # ------------------------------------------------------------

    resumen = (
        df.groupby("Categoría", as_index=False)
        .agg(
            Cantidad=("ID", "count"),
            Puntuación=("Puntaje_inherente", "sum")
        )
        .sort_values("Puntuación")
    )

    fig, axs = plt.subplots(
        1,
        2,
        figsize=(13, 5),
        layout="constrained"
    )

    colores_categoria = plt.cm.viridis(
        np.linspace(0.25, 0.85, len(resumen))
    )

    axs[0].barh(
        resumen["Categoría"],
        resumen["Puntuación"],
        color=colores_categoria
    )

    axs[0].set_title(
        "Suma de puntuaciones por categoría"
    )

    axs[0].set_xlabel(
        "Suma ordinal (depende de la cantidad de riesgos)"
    )

    conteo = (
        df["Nivel_inherente"]
        .value_counts()
        .reindex(list(COLORES), fill_value=0)
    )

    axs[1].pie(
        conteo,
        labels=conteo.index,
        colors=list(COLORES.values()),
        autopct=lambda p: f"{p:.0f}%" if p else "",
        startangle=90
    )

    axs[1].set_title(
        "Distribución de niveles inherentes"
    )

    fig.savefig(
        ruta / "04_perfil_riesgos.png"
    )

    plt.show()
    plt.close(fig)


# ================================================================
# 7. EXPORTACIÓN DE RESULTADOS
# ================================================================

def guardar(meta, df):
    """
    Crea una carpeta con los resultados del proyecto.

    Archivos generados:
    - Registro integral de riesgos en Excel.
    - Informe de interpretación gerencial.
    - Cuatro archivos PNG con las gráficas.

    La exportación permite utilizar los resultados
    como evidencia en el proyecto final.
    """

    ruta = Path.cwd() / (
        "Resultados_Riesgos_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
    )

    ruta.mkdir(
        parents=True,
        exist_ok=True
    )

    # Generación de gráficas.
    graficas(df, ruta)

    # Interpretación gerencial.
    texto = interpretar(df, meta)

    (ruta / "interpretacion_gerencial.txt").write_text(
        texto,
        encoding="utf-8"
    )

    # Tabla con el contexto de la empresa.
    contexto_df = pd.DataFrame([
        {"Campo": k, "Descripción": v}
        for k, v in meta.items()
    ])

    # Hoja metodológica para facilitar la interpretación.
    guia = pd.DataFrame({
        "Elemento": [
            "Probabilidad",
            "Impacto",
            "Puntuación",
            "Umbrales",
            "Residual",
            "Reducción_porcentual"
        ],
        "Definición": [
            "1 rara; 2 improbable; 3 posible; "
            "4 probable; 5 casi segura",

            "1 insignificante; 2 menor; 3 moderado; "
            "4 mayor; 5 severo",

            "Producto ordinal P × I, utilizado para priorizar",

            "Bajo 1–4; Moderado 5–9; "
            "Alto 10–16; Crítico 17–25",

            "Escenario estimado después de controles; "
            "requiere validación",

            "Variación relativa de puntuaciones; "
            "no reducción probabilística de pérdidas"
        ]
    })

    archivo_excel = ruta / "registro_integral_riesgos.xlsx"

    with pd.ExcelWriter(
        archivo_excel,
        engine="openpyxl"
    ) as writer:

        contexto_df.to_excel(
            writer,
            sheet_name="Contexto",
            index=False
        )

        df.to_excel(
            writer,
            sheet_name="Registro de riesgos",
            index=False
        )

        guia.to_excel(
            writer,
            sheet_name="Metodología",
            index=False
        )

        # Aplicación de formato a las hojas.
        from openpyxl.styles import Font, PatternFill

        for hoja in writer.book.worksheets:

            hoja.freeze_panes = "A2"
            hoja.auto_filter.ref = hoja.dimensions

            for columna in hoja.columns:

                letra = columna[0].column_letter

                ancho = max(
                    len(str(celda.value or ""))
                    for celda in columna[:min(len(columna), 30)]
                ) + 2

                hoja.column_dimensions[letra].width = min(
                    55,
                    max(15, ancho)
                )

            for celda in hoja[1]:

                celda.font = Font(
                    color="FFFFFF",
                    bold=True
                )

                celda.fill = PatternFill(
                    "solid",
                    fgColor="17365D"
                )

    # Presentación del informe en pantalla.
    print("\n")
    print(texto)

    print("\nARCHIVOS GENERADOS:")
    print(ruta.resolve())

    # Descarga opcional de todos los resultados en Colab.
    # La carpeta se comprime para facilitar su distribución.
    try:
        from google.colab import files
        import shutil

        archivo_zip = shutil.make_archive(
            str(ruta),
            "zip",
            root_dir=ruta
        )

        print("\nDescargando resultados comprimidos...")
        files.download(archivo_zip)

    except ImportError:
        # Fuera de Colab, los archivos permanecen
        # disponibles en la carpeta de resultados.
        pass


# ================================================================
# 8. EJECUCIÓN DEL LABORATORIO
# ================================================================

def main():
    """
    Ejecuta el flujo completo:

    1. Preguntas al equipo.
    2. Construcción del registro de riesgos.
    3. Generación de gráficas.
    4. Interpretación gerencial.
    5. Exportación de resultados.
    """

    contexto, datos = registrar()

    guardar(
        contexto,
        datos
    )


if __name__ == "__main__":

    try:
        main()

    except KeyboardInterrupt:
        print(
            "\nLa ejecución fue cancelada por el usuario."
        )
